# Tensor Shapes — see the padding tax

A short companion to the lesson [*Tensor Shapes*](https://lms-p-45c03.web.app/topics/math-infra/tensor-shapes/).
Compute the padding waste yourself and **chart** the "sawtooth" — zero at every multiple of 128, a
cliff just past it. **Runs on CPU**: just `numpy` + `matplotlib`, pure arithmetic, no GPU.

## 1. The rounding rule

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

T = 128                                  # MXU tile (256 on v6e/v7)
def pad(x, T=T): return int(np.ceil(x / T) * T)

for d in [128, 129, 256, 513, 512]:
    print(f"{d:>4} -> pads to {pad(d)}")

## 2. The bill on one matmul
A matmul `[M,K]·[K,N]` pays for the **padded** shape. Compare an aligned 512³ to a 129-wide one.

In [ ]:
def waste(M, K, N):
    Mp, Kp, Np = pad(M), pad(K), pad(N)
    flop = (Mp*Kp*Np) / (M*K*N) - 1                          # wasted compute
    mem  = (Mp*Kp + Kp*Np + Mp*Np) / (M*K + K*N + M*N) - 1   # wasted memory (A,B,out)
    return Mp, Kp, Np, flop*100, mem*100

for (M,K,N) in [(512,512,512), (129,512,512), (513,512,512)]:
    Mp,Kp,Np,fw,mw = waste(M,K,N)
    print(f"{M}x{K}x{N} -> {Mp}x{Kp}x{Np}   wasted compute {fw:4.0f}%   wasted memory {mw:4.0f}%")

## 3. The sawtooth of waste
Sweep one dimension and plot the wasted compute. It drops to **0 at every multiple of 128** and
**spikes just past one** — at 129 you nearly double the work. This is why "round to 128" is the rule.

In [ ]:
M = K = 512                              # keep the other dims aligned
Ns = np.arange(64, 1025)
w = np.array([waste(M, K, int(n))[3] for n in Ns])

plt.figure(figsize=(9, 4))
plt.plot(Ns, w, color="#dc2626", lw=2)
for x in range(128, 1025, 128):
    plt.axvline(x, ls=":", color="#0d9488", alpha=.4)
plt.annotate("129 -> 256\n+98%", xy=(129, 98), xytext=(220, 85),
             arrowprops=dict(arrowstyle="->", color="#64748b"), fontsize=9)
plt.xlabel("N (the swept dimension)"); plt.ylabel("wasted compute  (%)")
plt.title("Padding waste vs dimension — 0 on the 128 grid, a cliff just past it")
plt.grid(alpha=.2); plt.tight_layout(); plt.show()
print("teal dotted lines = multiples of 128 (zero waste). Land on them.")

## Put it together
1. **§1:** why does 129 pad all the way to 256, not 130? (Hint: it needs a *second* 128-tile.)
2. **§2:** the 129-wide matmul wastes ~98% compute and ~33% memory. Which is the one that OOMs your chip, and why?
3. **§3:** read the sawtooth — where are the safe dimensions, and where's the worst place to land?
4. You can't pick the model's dims. So what's *your* move when you spot a `128k+1` dimension on a job that's OOMing?

Back to the lesson → [Tensor Shapes](https://lms-p-45c03.web.app/topics/math-infra/tensor-shapes/)